In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import grover_operator
from qiskit_aer import AerSimulator
import math
import time
 
# ── Direction move gates
# (bit_b, bit_a):
# 00 = Up    (row -= 1)
# 01 = Right (col += 1)
# 10 = Down  (row += 1)
# 11 = Left  (col -= 1)
# ctrl_state rightmost character maps to the first ctrl qubit.
 
def add_up_move(qc, step, row, path):
    bit_a = path[step * 2]
    bit_b = path[step * 2 + 1]
    qc.mcx([bit_a, bit_b], row[0], ctrl_state='00')
    qc.mcx([bit_a, bit_b, row[0]], row[1], ctrl_state='100')
    qc.mcx([bit_a, bit_b, row[0], row[1]], row[2], ctrl_state='1100')
 
def add_down_move(qc, step, row, path):
    bit_a = path[step * 2]
    bit_b = path[step * 2 + 1]
    qc.mcx([bit_a, bit_b], row[0], ctrl_state='10')
    qc.mcx([bit_a, bit_b, row[0]], row[1], ctrl_state='010')
    qc.mcx([bit_a, bit_b, row[0], row[1]], row[2], ctrl_state='0010')
 
def add_left_move(qc, step, col, path):
    bit_a = path[step * 2]
    bit_b = path[step * 2 + 1]
    qc.mcx([bit_a, bit_b], col[0], ctrl_state='11')
    qc.mcx([bit_a, bit_b, col[0]], col[1], ctrl_state='111')
    qc.mcx([bit_a, bit_b, col[0], col[1]], col[2], ctrl_state='1111')
 
def add_right_move(qc, step, col, path):
    bit_a = path[step * 2]
    bit_b = path[step * 2 + 1]
    qc.mcx([bit_a, bit_b], col[0], ctrl_state='01')
    qc.mcx([bit_a, bit_b, col[0]], col[1], ctrl_state='001')
    qc.mcx([bit_a, bit_b, col[0], col[1]], col[2], ctrl_state='0001')
 
def undo_up_move(qc, step, row, path):
    bit_a = path[step * 2]
    bit_b = path[step * 2 + 1]
    qc.mcx([bit_a, bit_b, row[0], row[1]], row[2], ctrl_state='1100')
    qc.mcx([bit_a, bit_b, row[0]], row[1], ctrl_state='100')
    qc.mcx([bit_a, bit_b], row[0], ctrl_state='00')
 
def undo_down_move(qc, step, row, path):
    bit_a = path[step * 2]
    bit_b = path[step * 2 + 1]
    qc.mcx([bit_a, bit_b, row[0], row[1]], row[2], ctrl_state='0010')
    qc.mcx([bit_a, bit_b, row[0]], row[1], ctrl_state='010')
    qc.mcx([bit_a, bit_b], row[0], ctrl_state='10')
 
def undo_left_move(qc, step, col, path):
    bit_a = path[step * 2]
    bit_b = path[step * 2 + 1]
    qc.mcx([bit_a, bit_b, col[0], col[1]], col[2], ctrl_state='1111')
    qc.mcx([bit_a, bit_b, col[0]], col[1], ctrl_state='111')
    qc.mcx([bit_a, bit_b], col[0], ctrl_state='11')
 
def undo_right_move(qc, step, col, path):
    bit_a = path[step * 2]
    bit_b = path[step * 2 + 1]
    qc.mcx([bit_a, bit_b, col[0], col[1]], col[2], ctrl_state='0001')
    qc.mcx([bit_a, bit_b, col[0]], col[1], ctrl_state='001')
    qc.mcx([bit_a, bit_b], col[0], ctrl_state='01')
 
def row_col_ctrl_state(r, c):
    return (f"{(c>>2)&1}{(c>>1)&1}{(c>>0)&1}"
            f"{(r>>2)&1}{(r>>1)&1}{(r>>0)&1}")

# def add_bounds_check(qc, step, row, col, valid, path):
#     """
#     (bit_b, bit_a):
#     00 = Up    (row -= 1)
#     01 = Right (col += 1)
#     10 = Down  (row += 1)
#     11 = Left  (col -= 1)
#     ctrl_state rightmost character maps to the first ctrl qubit.
#     """
    
#     bit_a = path[step * 2]
#     bit_b = path[step * 2 + 1]
#     valid_bit = valid[step]

#     qc.x(valid_bit)
    
#     #1 if row == 0 and move is up then invalid
#     qc.mcx([bit_a, bit_b, row[0], row[1], row[2]], valid_bit, ctrl_state="00000")
    
#     #2 if row == len(maze) and move is down then invalid, ctrl state only needs to invert move as is backwards ordering
#     qc.mcx([bit_a, bit_b, row[0], row[1], row[2]], valid_bit, ctrl_state="11001")
    
#     #3 if col == 0 and move is left then invalid
#     qc.mcx([bit_a, bit_b, col[0], col[1], col[2]], valid_bit, ctrl_state="00011")
    
#     #4 if row == len(maze[0]) and move is right then invalid
#     qc.mcx([bit_a, bit_b, col[0], col[1], col[2]], valid_bit, ctrl_state="11010")
    
#     qc.x(valid_bit)

# def undo_bounds_check(qc, step, row, col, valid, path):
#     """
#     mcx gates are so sick
#     """
    
#     bit_a = path[step * 2]
#     bit_b = path[step * 2 + 1]
#     valid_bit = valid[step]

#     qc.x(valid_bit)
    
#     #4 if row == 3 and move is right then invalid
#     qc.mcx([bit_a, bit_b, col[0], col[1], col[2]], valid_bit, ctrl_state="11010")
    
#     #3 if col == 0 and move is left then invalid
#     qc.mcx([bit_a, bit_b, col[0], col[1], col[2]], valid_bit, ctrl_state="00011")
    
#     #2 if row == 3 and move is down then invalid, ctrl state only needs to invert move as is backwards ordering
#     qc.mcx([bit_a, bit_b, row[0], row[1], row[2]], valid_bit, ctrl_state="11001")
    
#     #1 if row == 0 and move is up then invalid
#     qc.mcx([bit_a, bit_b, row[0], row[1], row[2]], valid_bit, ctrl_state="00000")
    
#     qc.x(valid_bit)

    
 
# ── Oracle ───────────────────────────────────────────────────────────────────
 
def build_oracle(maze, end, num_steps):
    path  = QuantumRegister(num_steps * 2, name='path')
    row   = QuantumRegister(3,             name='row')
    col   = QuantumRegister(3,             name='col')
    valid = QuantumRegister(num_steps,     name='valid')
    flag  = QuantumRegister(1,             name='flag')
    qc    = QuantumCircuit(path, row, col, valid, flag)
 
    walls = [(r, c) for r in range(len(maze))
             for c in range(len(maze[0])) if maze[r][c] == 1]
    
    #going forwards: (update position, mark wall hits), stop out of bounds or wrap arounds
    for step in range(num_steps):
        add_left_move(qc, step, col, path)  #1
        add_down_move(qc, step, row, path)  #2
        add_right_move(qc, step, col, path) #3
        add_up_move(qc, step, row, path)    #4

        for (r, c) in walls:
            qc.mcx(row[:] + col[:], valid[step], ctrl_state=row_col_ctrl_state(r, c))
 
    # check goal: invert valid = 1 = no walls hit
    for step in range(num_steps):
        qc.x(valid[step])
 
    end_r, end_c = end
    ctrl_state_end = (f"{(end_c>>2)&1}{(end_c>>1)&1}{(end_c>>0)&1}"
                      f"{(end_r>>2)&1}{(end_r>>1)&1}{(end_r>>0)&1}"
                      + '1' * num_steps)
    qc.mcx(valid[:] + row[:] + col[:], flag[0], ctrl_state=ctrl_state_end)
 
    for step in range(num_steps):
        qc.x(valid[step])
 
    # Phase flip on flag qubit
    qc.z(flag[0])
 
    # Uncompute flag
    for step in range(num_steps):
        qc.x(valid[step])
    qc.mcx(valid[:] + row[:] + col[:], flag[0], ctrl_state=ctrl_state_end)
    for step in range(num_steps):
        qc.x(valid[step])
 
    # Reverse: uncompute valid and position
    for step in reversed(range(num_steps)):
        for (r, c) in walls: #walls
            qc.mcx(row[:] + col[:], valid[step], ctrl_state=row_col_ctrl_state(r, c))

        undo_up_move(qc, step, row, path)    #4
        undo_right_move(qc, step, col, path) #3
        undo_down_move(qc, step, row, path)  #2
        undo_left_move(qc, step, col, path)  #1
 
    return qc
 
 
# ── Result decoding ──────────────────────────────────────────────────────────
 
DIRECTION_MAP   = {0b00: 'U', 0b10: 'D', 0b11: 'L', 0b01: 'R'}
DIRECTION_DELTA = {'U': (-1, 0), 'D': (1, 0), 'L': (0, -1), 'R': (0, 1)}
 
def decode_path(bitstring, num_steps):
    """Convert bitstrings to a list of directions"""
    path_bits = bitstring[-(num_steps * 2):]
    directions = []
    for step in range(num_steps):
        bit_a = int(path_bits[-(step * 2 + 1)])
        bit_b = int(path_bits[-(step * 2 + 2)])
        directions.append(DIRECTION_MAP.get((bit_b << 1) | bit_a))
    return directions
 
def evaluate_path(directions, start, end, maze):
    """
    Checker to measure success of algorithm
    against classical method
    """
    rows, cols = len(maze), len(maze[0])
    pos = list(start)
    trace = [tuple(pos)]
    hit_wall = False
    for d in directions:
        dr, dc = DIRECTION_DELTA[d]
        pos[0] += dr; pos[1] += dc
        oob = not (0 <= pos[0] < rows and 0 <= pos[1] < cols)
        if oob or maze[pos[0]][pos[1]] == 1:
            hit_wall = True
        trace.append(tuple(pos))
    return {'trace': trace, 'hit_wall': hit_wall,
            'at_goal': tuple(pos) == tuple(end),
            'is_valid': not hit_wall and tuple(pos) == tuple(end)}
 
 
# ── Main runner ──────────────────────────────────────────────────────────────
def run_grover_maze_solver(maze, start, end, num_steps, shots=1024, num_iterations=None):
    
    num_path   = num_steps * 2
    total_q    = num_path + 3 + 3 + num_steps + 1
    N          = 2 ** num_path
 
    if num_iterations is None:
        num_iterations = round(math.pi / 4 * math.sqrt(N))
    else:
        num_iterations = max(num_iterations, round(math.pi / 4 * math.sqrt(N)))
        
    print("=" * 50 + "\n Quantum walk using grovers algorithm \n" + "=" * 50)
    print(f"Search space:      2^{num_path} = {N}")
    print(f"Maze size:         {len(maze)} x {len(maze[0])}")
    print(f"Start:             {start}")
    print(f"End:               {end}")
    print(f"Steps:             {num_steps}")
    print(f"Shots:             {shots}")
    print(f"Grover iterations: {num_iterations}")
    print(f"Path qubits:       {num_path}")
    print(f"Total qubits:      {total_q}")
    print(f"memory:           ~{(2**total_q) * 16 /1e9:.3f} GB") #16 as double float precision
 
    print("Maze layout")
    for ri, row_data in enumerate(maze):
        row_str = '  '
        for ci, cell in enumerate(row_data):
            if (ri, ci) == tuple(start):
                row_str += 'S '
            elif (ri, ci) == tuple(end):
                row_str += 'E '
            elif cell == 1:
                row_str += '█ '
            else:
                row_str += '. '
        print(row_str)
 
    # Adjust goal coords relative to start position, as might not always be 0 in future
    effective_end = (end[0] - start[0], end[1] - start[1])
    oracle_qc = build_oracle(maze, effective_end, num_steps)
    grover_op = grover_operator(
        oracle_qc,
        reflection_qubits=list(range(num_path))
    )
 
    #Build full Grover circuit
    clbits  = ClassicalRegister(num_path, name='c')
    full_qc = QuantumCircuit(total_q)
    full_qc.add_register(clbits)
    full_qc.h(range(num_path)) #initial superposition, randomises probabilities
 
    for _ in range(num_iterations):
        full_qc.compose(grover_op, inplace=True)# oracle and diffuser
 
    full_qc.measure(range(num_path), clbits)
 
    print(f"Circuit depth: {full_qc.depth()}, gate count: {full_qc.size()}")
    print("..Running simulation..")
 
    t0     = time.time()
    sim    = AerSimulator(method='statevector')
    counts = sim.run(full_qc, shots=shots).result().get_counts()
    print(f"Simulation completed in: {time.time()-t0:.1f}s\n")
 
    sorted_counts = sorted(counts.items(), key=lambda x: -x[1])
    results = []
    for bitstring, count in sorted_counts:
        dirs = decode_path(bitstring, num_steps)
        info = evaluate_path(dirs, start, end, maze)
        results.append({'bitstring': bitstring, 'count': count,
                        'directions': dirs, **info})
 
    w = num_path + 2
    print(f"{'Rank':<8} {'Bitstring':<{w}} {'Shots':>6}  {'Dirs':<12} {'Trace':<48} Status")
    print('-' * (w + 80))
    for rank, r in enumerate(results[:12], 1):
        dirs_str  = ''.join(r['directions'])
        trace_str = '->'.join(str(p) for p in r['trace'])
        status    = '✓ solution' if r['is_valid'] else ''
        print(f"{rank:<5} {r['bitstring']:<{w}} {r['count']:>6}  "
              f"{dirs_str:<12} {trace_str:<48} {status}")
 
    sol_shots = sum(r['count'] for r in results if r['is_valid'])
    print(f"\nSolution shots: {sol_shots}/{shots} ({100*sol_shots/shots:.1f}%)")
 
    print("\nsolution paths:")
    seen = set()
    for r in results:
        if r['is_valid']:
            key = tuple(r['directions'])
            if key not in seen:
                print(f"  {''.join(r['directions'])}  →  "+ '  ->  '.join(str(p) for p in r['trace']))
                seen.add(key)
                
    return results
 
 
# ── Configuration ───


In [4]:
if __name__ == '__main__':
 
    maze = [
    [0, 0, 1, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 1, 1, 1],
    ]
    
    START     = (0, 0)
    END       = (3, 0)
    NUM_STEPS = 5
 
    run_grover_maze_solver(
        maze      = maze,
        start     = START,
        end       = END,
        num_steps = NUM_STEPS,
        shots     = 1024,
        num_iterations = 5
    )

 Quantum walk using grovers algorithm 
Search space:      2^10 = 1024
Maze size:         4 x 4
Start:             (0, 0)
End:               (3, 0)
Steps:             5
Shots:             1024
Grover iterations: 25
Path qubits:       10
Total qubits:      22
memory:           ~0.067 GB
Maze layout
  S . █ . 
  █ . █ . 
  . . . . 
  E █ █ █ 
Circuit depth: 4778, gate count: 6170
..Running simulation..
Simulation completed in: 79.8s

Rank  Bitstring     Shots  Dirs         Trace                                            Status
--------------------------------------------------------------------------------------------
1     1101001111        6  LLURL        (0, 0)->(0, -1)->(0, -2)->(-1, -2)->(-1, -1)->(-1, -2) 
2     1100001100        5  ULUUL        (0, 0)->(-1, 0)->(-1, -1)->(-2, -1)->(-3, -1)->(-3, -2) 
3     0011110000        5  UULLU        (0, 0)->(-1, 0)->(-2, 0)->(-2, -1)->(-2, -2)->(-3, -2) 
4     0100101101        5  RLDUR        (0, 0)->(0, 1)->(0, 0)->(1, 0)->(0, 0)->(0, 1) 